# Bloque 5. JSON y NDJSON

**Modulo ETCDFM (5104) - UD2: Formatos de fichero de datos**

---

**JSON** (*JavaScript Object Notation*) es el formato de intercambio de datos predominante en APIs REST, configuracion de aplicaciones y almacenamiento de documentos. Su variante **NDJSON** (*Newline-Delimited JSON*) o *JSON Lines* permite procesar datasets grandes registro a registro sin cargarlos completamente en memoria. Juntos son la columna vertebral de los pipelines de datos modernos.


## Contenidos

1. JSON: estructura, tipos y sintaxis
2. Libreria estandar `json`: `load`, `dump`, `loads`, `dumps`
3. JSON y pandas: `read_json`, `to_json`, parametro `orient`
4. JSON anidado: `pd.json_normalize`
5. NDJSON / JSON Lines: formato registro a registro
6. NDJSON en pandas: `read_json(lines=True)` y `to_json(lines=True)`
7. JSON desde APIs REST publicas
8. Comparativa: JSON vs XML vs CSV


## 5.1 JSON: estructura y tipos de datos

JSON es texto plano con estructura de **objeto** (diccionario) o **array** (lista), anidables libremente:

```json
{
  "titulo": "Inception",
  "ano": 2010,
  "disponible": true,
  "director": null,
  "generos": ["Ciencia Ficcion", "Thriller"],
  "puntuaciones": {
    "imdb": 8.8,
    "rotten_tomatoes": 87
  }
}
```

| Tipo JSON | Equivalente Python | Ejemplo |
|---|---|---|
| `string` | `str` | `"hola"` |
| `number` | `int` / `float` | `42`, `3.14` |
| `boolean` | `bool` | `true`, `false` |
| `null` | `None` | `null` |
| `array` | `list` | `[1, 2, 3]` |
| `object` | `dict` | `{"clave": "valor"}` |


![Estructura de un objeto JSON: tipos y anidamiento](imagenes/b5_objeto_json.png)

## 5.2 Libreria estandar `json`

```python
import json

# Leer desde fichero  ->  Python (dict / list)
with open('datos.json', 'r', encoding='utf-8') as f:
    datos = json.load(f)

# Escribir a fichero  <-  Python (dict / list)
with open('salida.json', 'w', encoding='utf-8') as f:
    json.dump(datos, f, ensure_ascii=False, indent=2)

# Desde/hacia string (sin fichero)
texto = json.dumps(datos, ensure_ascii=False)
datos = json.loads(texto)
```

**Parametros utiles de `json.dump` / `json.dumps`:**
- `indent=2` — sangria para JSON legible por humanos
- `ensure_ascii=False` — permite caracteres no ASCII (tildes, etc.)
- `sort_keys=True` — ordenar claves alfabeticamente
- `default=str` — serializar tipos no soportados (p.ej. `datetime`) como string


In [2]:
import json

# Cargar el catalogo de peliculas
with open('datos/peliculas.json', 'r', encoding='utf-8') as f:
    peliculas = json.load(f)

print(f'Tipo: {type(peliculas)}  |  N. peliculas: {len(peliculas)}')
print(f'Primera pelicula (claves): {list(peliculas[0].keys())}')
print()

# Acceso natural como diccionario Python
p = peliculas[0]
print(f"Titulo:       {p['titulo']}")
print(f"Generos:      {p['generos']}")
print(f"IMDB:         {p['puntuaciones']['imdb']}")
print(f"Oscars:       {p['premios']['oscar_ganados']} ganados / {p['premios']['oscar_nominaciones']} nominaciones")

# Filtrar y calcular
disponibles = [p for p in peliculas if p['disponible_streaming']]
print(f'\nPeliculas disponibles en streaming: {len(disponibles)}')

nota_media = sum(p['puntuaciones']['imdb'] for p in peliculas) / len(peliculas)
print(f'Nota IMDB media: {nota_media:.2f}')

# Guardar subset a JSON
top = sorted(peliculas, key=lambda p: p['puntuaciones']['imdb'], reverse=True)[:5]
with open('/tmp/top5_peliculas.json', 'w', encoding='utf-8') as f:
    json.dump(top, f, ensure_ascii=False, indent=2)

print(f'\nTop 5 por IMDB guardado en /tmp/top5_peliculas.json')
for p in top:
    print(f"  {p['puntuaciones']['imdb']:.1f}  {p['titulo']} ({p['ano']})")


Tipo: <class 'list'>  |  N. peliculas: 30
Primera pelicula (claves): ['id', 'titulo', 'director', 'ano', 'generos', 'duracion_min', 'puntuaciones', 'recaudacion_millon_usd', 'idioma_original', 'disponible_streaming', 'premios']

Titulo:       Inception
Generos:      ['Ciencia Ficcion', 'Thriller', 'Accion']
IMDB:         8.7
Oscars:       4 ganados / 1 nominaciones

Peliculas disponibles en streaming: 20
Nota IMDB media: 8.36

Top 5 por IMDB guardado en /tmp/top5_peliculas.json
  9.3  Alien (1979)
  9.2  The Usual Suspects (1995)
  9.2  Get Out (2017)
  9.1  The Shawshank Redemption (1994)
  9.1  Mad Max: Fury Road (2015)


## 5.3 JSON y pandas: `read_json` y `orient`

El parametro `orient` indica como esta organizado el JSON:

| `orient` | Estructura JSON | Ejemplo |
|---|---|---|
| `'records'` | Lista de dicts | `[{col1:v, col2:v}, ...]` |
| `'columns'` | Dict de columnas (por defecto) | `{col1: {0:v, 1:v}, col2: ...}` |
| `'index'` | Dict de indices | `{0: {col1:v}, 1: ...}` |
| `'values'` | Array de valores | `[[v1, v2], [v1, v2]]` |
| `'split'` | Dict con `index`, `columns`, `data` | Compacto |
| `'table'` | Con schema de tipos incluido | Para round-trip fiel |

```python
# Lista de dicts -> el mas comun en APIs
df = pd.read_json('datos/peliculas.json', orient='records')

# Exportar
df.to_json('salida.json', orient='records', force_ascii=False, indent=2)
```

> Cuando el fichero es una lista de objetos JSON, `orient='records'` es el correcto y suele inferirse automaticamente.


In [3]:
import pandas as pd

# Leer lista de peliculas
df = pd.read_json('datos/peliculas.json', orient='records')
print(f'Shape: {df.shape}')
print(df.dtypes)
print()

# Los campos anidados quedan como objetos Python en la celda
print('Columna generos (primeras 3 filas):')
print(df['generos'].head(3))
print()
print('Columna puntuaciones (dict anidado):')
print(df['puntuaciones'].head(3))

# Comparar orient
print('\n--- Distintos orient al exportar ---')
df_mini = df[['titulo', 'ano', 'director']].head(2)

print('\norient=records:')
print(df_mini.to_json(orient='records', force_ascii=False))

print('\norient=split:')
print(df_mini.to_json(orient='split', force_ascii=False))

print('\norient=index:')
print(df_mini.to_json(orient='index', force_ascii=False))


Shape: (30, 11)
id                          int64
titulo                     object
director                   object
ano                         int64
generos                    object
duracion_min                int64
puntuaciones               object
recaudacion_millon_usd    float64
idioma_original            object
disponible_streaming         bool
premios                    object
dtype: object

Columna generos (primeras 3 filas):
0    [Ciencia Ficcion, Thriller, Accion]
1              [Ciencia Ficcion, Accion]
2     [Ciencia Ficcion, Drama, Aventura]
Name: generos, dtype: object

Columna puntuaciones (dict anidado):
0    {'imdb': 8.7, 'rotten_tomatoes': 93, 'metacrit...
1    {'imdb': 7.7, 'rotten_tomatoes': 71, 'metacrit...
2    {'imdb': 8.5, 'rotten_tomatoes': 92, 'metacrit...
Name: puntuaciones, dtype: object

--- Distintos orient al exportar ---

orient=records:
[{"titulo":"Inception","ano":2010,"director":"Christopher Nolan"},{"titulo":"The Matrix","ano":1999,"director":"The

## 5.4 JSON anidado: `pd.json_normalize`

Cuando el JSON tiene campos anidados (dicts dentro de dicts, o listas), `pd.json_normalize` los **aplana** a columnas planas:

```python
from pandas import json_normalize

# Aplanar dicts anidados
df = json_normalize(datos,
                    sep='.')          # separador para las columnas anidadas

# Con listas de registros anidadas (record_path)
df = json_normalize(
    datos,
    record_path=['historial_compras'],     # lista anidada a aplanar
    meta=['id', 'nombre', 'edad'],         # campos del padre a incluir
    sep='.'
)
```

El resultado es un DataFrame donde cada campo anidado se convierte en columna con nombre `padre.hijo` (segun el `sep`).


In [4]:
import json
from pandas import json_normalize

with open('datos/usuarios_anidado.json', 'r', encoding='utf-8') as f:
    usuarios = json.load(f)

# Aplanar un solo nivel: dicts dentro de dicts
df = json_normalize(usuarios, sep='.')
print(f'Shape: {df.shape}')
print('Columnas anidadas aplanadas:')
cols_anidadas = [c for c in df.columns if '.' in c]
print(cols_anidadas)
print()
print(df[['nombre', 'direccion.ciudad', 'preferencias.notificaciones_email']].head(4))

# Explotar la lista historial_compras: una fila por compra
df_compras = json_normalize(
    usuarios,
    record_path=['historial_compras'],
    meta=['id', 'nombre', 'edad'],
    sep='.'
)
print(f'\nDataFrame de compras (una fila por pedido): {df_compras.shape}')
print(df_compras[['nombre', 'fecha', 'categoria', 'total_eur']].head(6))
print()
print('Gasto total por usuario (top 5):')
print(df_compras.groupby('nombre')['total_eur'].sum()
               .sort_values(ascending=False).head(5).round(2))


Shape: (20, 12)
Columnas anidadas aplanadas:
['direccion.calle', 'direccion.ciudad', 'direccion.codigo_postal', 'direccion.pais', 'preferencias.categorias_favoritas', 'preferencias.notificaciones_email']

          nombre direccion.ciudad  preferencias.notificaciones_email
0     Ana Garcia            Palma                               True
1   Carlos Lopez          Sevilla                               True
2   Maria Torres            Palma                               True
3  Pedro Sanchez        Barcelona                              False

DataFrame de compras (una fila por pedido): (63, 8)
         nombre       fecha     categoria  total_eur
0    Ana Garcia  2024-02-24  Alimentacion     335.70
1    Ana Garcia  2024-03-25  Alimentacion     178.03
2    Ana Garcia  2024-02-07        Libros     339.52
3  Carlos Lopez  2024-01-09      Deportes      37.77
4  Carlos Lopez  2024-02-21   Electronica     330.43
5  Maria Torres  2024-02-10        Libros     103.98

Gasto total por usuario (

## 5.5 NDJSON / JSON Lines: registro a registro

**NDJSON** (*Newline-Delimited JSON*) = un objeto JSON completo por linea:

```
{"timestamp": "2024-01-01T08:23:00", "usuario_id": 5, "accion": "login"}
{"timestamp": "2024-01-01T08:25:00", "usuario_id": 5, "accion": "compra", "datos": {"total_eur": 89.99}}
{"timestamp": "2024-01-01T08:31:00", "usuario_id": 3, "accion": "busqueda"}
```

**Ventajas sobre JSON estandar:**
- Se puede leer **linea a linea** sin cargar todo el fichero en memoria.
- Ideal para **logs**, **eventos** y **streaming** de datos.
- Compatible con herramientas de linea de comandos (`grep`, `jq`, `awk`).
- Cada linea es valida de forma independiente (tolerante a corrupcion parcial).

**Extensiones comunes:** `.ndjson`, `.jsonl`, `.jl`

```python
# Leer NDJSON linea a linea (streaming)
with open('eventos.ndjson') as f:
    for linea in f:
        evento = json.loads(linea)
        # procesar evento...
```


In [5]:
import json
from collections import Counter

# Leer NDJSON linea a linea (streaming: no carga todo en memoria)
acciones = Counter()
dispositivos = Counter()
total_compras = 0.0
n_lineas = 0

with open('datos/eventos.ndjson', 'r', encoding='utf-8') as f:
    for linea in f:
        evento = json.loads(linea.strip())
        acciones[evento['accion']] += 1
        dispositivos[evento['dispositivo']] += 1
        if evento['accion'] == 'compra' and evento['datos']:
            total_compras += evento['datos'].get('total_eur', 0)
        n_lineas += 1

print(f'Total eventos procesados: {n_lineas}')
print(f'Total compras (EUR): {total_compras:,.2f}')
print(f'\nAcciones:')
for accion, cnt in acciones.most_common():
    print(f'  {accion:<22} {cnt:>4}')
print(f'\nDispositivos: {dict(dispositivos)}')

# Escribir NDJSON: exportar solo eventos de compra
with open('datos/eventos.ndjson', 'r') as fin, \
     open('/tmp/compras.ndjson', 'w') as fout:
    for linea in fin:
        ev = json.loads(linea)
        if ev['accion'] == 'compra':
            fout.write(linea)

import os
n_compras = sum(1 for _ in open('/tmp/compras.ndjson'))
print(f'\nEventos de compra exportados: {n_compras}')


Total eventos procesados: 500
Total compras (EUR): 14,168.54

Acciones:
  login                    57
  compra                   56
  anadir_carrito           51
  valoracion               50
  eliminar_carrito         50
  busqueda                 49
  descarga_factura         49
  cambio_perfil            48
  vista_producto           47
  logout                   43

Dispositivos: {'tablet': 172, 'desktop': 164, 'mobile': 164}

Eventos de compra exportados: 56


## 5.6 NDJSON con pandas

Con `lines=True`, pandas trata cada linea como un registro JSON independiente:

```python
import pandas as pd

# Leer NDJSON completo
df = pd.read_json('datos/eventos.ndjson', lines=True)

# Lectura por chunks (ficheros muy grandes)
for chunk in pd.read_json('datos/eventos.ndjson',
                           lines=True, chunksize=100):
    # procesar chunk...
    pass

# Exportar a NDJSON
df.to_json('salida.ndjson',
            orient='records',
            lines=True,
            force_ascii=False)
```

> `orient='records'` + `lines=True` es la combinacion correcta para NDJSON. Si solo se usa `lines=True` sin `orient`, pandas usa `orient='columns'` por defecto, que no produce NDJSON valido.


In [6]:
import pandas as pd

# Leer NDJSON completo a DataFrame
df = pd.read_json('datos/eventos.ndjson', lines=True)
print(f'Shape: {df.shape}')
print(df.dtypes)
print()
print(df[['timestamp', 'usuario_id', 'accion', 'dispositivo']].head(5))

# Analisis
print('\nEventos por accion:')
print(df['accion'].value_counts())

print('\nEventos por dispositivo:')
print(df['dispositivo'].value_counts())

# Lectura por chunks (util con ficheros de GB)
print('\n--- Lectura por chunks (chunksize=100) ---')
n_compras = 0
total_eur  = 0.0
for chunk in pd.read_json('datos/eventos.ndjson',
                           lines=True, chunksize=100):
    compras = chunk[chunk['accion'] == 'compra']
    n_compras += len(compras)
    for datos in compras['datos'].dropna():
        if isinstance(datos, dict):
            total_eur += datos.get('total_eur', 0)
print(f'Compras encontradas: {n_compras}  Total: {total_eur:,.2f} EUR')

# Exportar subset a NDJSON
df_logins = df[df['accion'] == 'login'][['timestamp', 'usuario_id', 'dispositivo', 'sistema']]
df_logins.to_json('datos/logins.ndjson', orient='records', lines=True, force_ascii=False)
print(f'\nlogins.ndjson exportado: {len(df_logins)} registros')


Shape: (500, 8)
timestamp      datetime64[ns]
usuario_id              int64
session_id             object
accion                 object
dispositivo            object
sistema                object
ip                     object
datos                  object
dtype: object

            timestamp  usuario_id          accion dispositivo
0 2024-01-01 02:48:00          10  vista_producto      tablet
1 2024-01-01 05:29:00          18      valoracion     desktop
2 2024-01-01 05:40:00          15  vista_producto     desktop
3 2024-01-01 06:40:00           3          compra      tablet
4 2024-01-01 08:48:00           4  anadir_carrito      tablet

Eventos por accion:
accion
login               57
compra              56
anadir_carrito      51
valoracion          50
eliminar_carrito    50
descarga_factura    49
busqueda            49
cambio_perfil       48
vista_producto      47
logout              43
Name: count, dtype: int64

Eventos por dispositivo:
dispositivo
tablet     172
desktop    164
mobil

## 5.7 JSON desde APIs REST publicas

La mayoria de APIs web devuelven JSON. Las opciones en Python:

```python
# Con urllib (stdlib, sin dependencias)
import urllib.request, json

url = 'https://api.open-meteo.com/v1/forecast?latitude=40.42&longitude=-3.70&current_weather=true'
with urllib.request.urlopen(url) as resp:
    datos = json.load(resp)

# Con requests (libreria externa, mas comoda)
import requests

resp = requests.get(url)
resp.raise_for_status()    # lanza excepcion si HTTP != 200
datos = resp.json()        # decodifica automaticamente
```

**APIs publicas sin autenticacion utiles para practicas:**
- **Open-Meteo** `api.open-meteo.com` — prediccion meteorologica
- **Open Library** `openlibrary.org/api` — datos de libros
- **REST Countries** `restcountries.com` — datos de paises
- **PokeAPI** `pokeapi.co` — datos de Pokemon (JSON complejo, bueno para json_normalize)


In [7]:
import urllib.request, json
import pandas as pd

# Open-Meteo: temperatura horaria en Madrid los ultimos 7 dias
url = ('https://api.open-meteo.com/v1/forecast'
       '?latitude=40.42&longitude=-3.70'
       '&hourly=temperature_2m,precipitation'
       '&past_days=7&forecast_days=0'
       '&timezone=Europe%2FMadrid')

try:
    with urllib.request.urlopen(url, timeout=10) as resp:
        datos = json.load(resp)

    print('Claves de la respuesta:', list(datos.keys()))
    print('Claves de hourly:', list(datos['hourly'].keys()))

    # Construir DataFrame con los datos horarios
    df_meteo = pd.DataFrame({
        'hora':        pd.to_datetime(datos['hourly']['time']),
        'temp_c':      datos['hourly']['temperature_2m'],
        'precip_mm':   datos['hourly']['precipitation'],
    })
    print(f'\nDatos meteorologicos Madrid: {df_meteo.shape}')
    print(df_meteo.tail(6))

    print(f'\nTemp. media: {df_meteo["temp_c"].mean():.1f} C')
    print(f'Temp. max:   {df_meteo["temp_c"].max():.1f} C')
    print(f'Precipitacion total: {df_meteo["precip_mm"].sum():.1f} mm')

    # Guardar en NDJSON
    df_meteo.to_json('datos/meteo_madrid.ndjson',
                     orient='records', lines=True, force_ascii=False,
                     date_format='iso')
    print(f'Guardado en datos/meteo_madrid.ndjson')

except Exception as e:
    print(f'Error al acceder a la API: {e}')
    print('(Ejecutar con conexion a internet activa)')


Claves de la respuesta: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly']
Claves de hourly: ['time', 'temperature_2m', 'precipitation']

Datos meteorologicos Madrid: (168, 3)
                   hora  temp_c  precip_mm
162 2026-08-21 18:00:00    30.2        0.0
163 2026-08-21 19:00:00    29.9        0.0
164 2026-08-21 20:00:00    29.4        0.0
165 2026-08-21 21:00:00    28.6        0.0
166 2026-08-21 22:00:00    27.1        0.0
167 2026-08-21 23:00:00    25.7        0.0

Temp. media: 27.5 C
Temp. max:   38.1 C
Precipitacion total: 0.4 mm
Guardado en datos/meteo_madrid.ndjson


## 5.8 Comparativa: JSON vs XML vs CSV

| Caracteristica | CSV | JSON | XML |
|---|---|---|---|
| Estructura | Plana (tabular) | Jerarquica | Jerarquica |
| Legibilidad humana | Alta | Alta | Media |
| Tamaño relativo | Pequeno | Medio | Grande |
| Esquema formal | No | JSON Schema (opcional) | XSD / DTD |
| Tipos de datos | No (todo texto) | Si (6 tipos nativos) | No (todo texto con atributos) |
| Anidamiento | No | Si (ilimitado) | Si (ilimitado) |
| Namespaces | No | No | Si |
| Uso principal | Datos tabulares, Excel | APIs REST, config, NoSQL | Sistemas empresariales, SOAP |
| Soporte pandas | `read_csv` | `read_json` | `read_xml` |

**Regla rapida:**
si los datos son planos -> CSV; si son jerarquicos y van a una API -> JSON; si requieren esquema formal o integracion empresarial -> XML.


![¿Cuándo usar cada formato de texto?](imagenes/b5_csv_json_xml.png)

## Resumen del bloque

**`json` (stdlib)** — leer/escribir JSON con `load`/`dump` (ficheros) y `loads`/`dumps` (strings). Parametros clave: `indent`, `ensure_ascii=False`, `default=str`.

**pandas** — `read_json(orient=...)` para leer, `to_json(orient=..., force_ascii=False)` para exportar. El `orient` mas comun con APIs es `'records'`.

**`pd.json_normalize`** — aplana JSON anidado a columnas planas. `record_path` para explotar listas anidadas con campos del padre via `meta`.

**NDJSON / JSON Lines** — un objeto JSON por linea. Se lee linea a linea con `json.loads` o con `pd.read_json(lines=True)`. Ideal para logs, eventos y streaming.

**APIs REST** — `urllib.request.urlopen` (stdlib) o `requests.get(...).json()`. El resultado es un dict/list Python listo para `json_normalize` o para construir un DataFrame.

---

*Con este bloque se completan los cinco bloques de contenido de la UD2. Siguientes pasos: ejercicios y miniproyecto.*
